
# Qwen3-VL Vision Scene Enrichment — GPU Validation

Run this notebook on a **GPU runtime** (T4/A100) with a **real movie you
legally own** to prove the new *vision* stage of the Movie Intelligence Layer:
per-scene **keyframes → Qwen3-VL → visual scene cards**.

### What is under test (the NEW vision enrichment)
```
movie -> (existing) WhisperX transcript + PySceneDetect scenes
      -> keyframes per scene (ffmpeg, one frame mid-scene)
      -> Qwen2.5-VL / Qwen3-VL (AutoProcessor + AutoModel, sdpA, VRAM-aware)
      -> story fields: location / actions / visual_description / themes / mood
      -> movie_index.json (vision-filled story cards + per-field provenance)
```
This fills the last honest gap in the previous milestone: `location`,
`actions`, `visual_description`, `themes`, and `mood` were `None` + flagged
`unavailable (vision/LLM)`; this run proves they can come from a real visual
model on CUDA.

### Strict modes (no silent fallbacks)
- `REQUIRE_REAL_VISION=true` — the enricher must be the real VL model; a
  heuristic fallback (fields left None) is a hard error.
- `VISION_ENRICHER=qwen3vl` — selects the Qwen3VLEnricher behind the
  `SceneEnricher` provider interface (same code the `MovieAnalyzer` uses).

### Movie is never committed
Supply a **Google Drive share link** (`MOVIE_URL`), a path on the mounted
Drive, or upload a file. Optionally `TRIM_TO_SEC` to a short section first:
**60–180s is plenty** to validate vision enrichment cheaply.

### Artifacts to inspect after the run
`movie_index.json` (vision-filled `story` per scene + `provenance`),
`scene_index_v2.json` (versioned enriched scene index),
`semantic_index.json`, `movie_memory/` (director-facing bundle),
`scenes/keyframes/*.jpg`, `reports/movie_understanding_report.md`,
`reports/retrieval_evaluation.json` + `.md` (natural-language retrieval eval).



### Cell 1 — Runtime & system packages (base setup)
Installs FFmpeg, a consistent CUDA PyTorch, Transformers, Whisper and
PySceneDetect from `scripts/colab_setup.sh`. Set `REPO_URL` / `BRANCH` to your
fork/branch if needed, and supply the movie via `MOVIE_URL` or `MOVIE_PATH`.


In [ ]:

# @title 1) Setup: system + repo + base deps
import os, sys, subprocess

REPO_URL = "https://github.com/asdfhgds/automovies.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MOVIE_URL = ""  # @param {type:"string"} Google Drive share link (e.g. https://drive.google.com/file/d/xxx/view)
MOVIE_PATH = ""  # @param {type:"string"} Drive/local path (used if MOVIE_URL is empty)

def sh(cmd, **kw):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    sys.stdout.write(p.stdout or "")
    if p.returncode != 0:
        sys.stdout.write(p.stderr or "")
        raise SystemExit(f"command failed ({p.returncode}): {cmd}\n--- tail ---\n{(p.stderr or '')[-4000:]}")

# Always work from an absolute, deterministic location so re-running this cell
# never nests extra copies of the repo.
ROOT = "/content"
REPO_DIR = os.path.join(ROOT, "automovies")
os.chdir(ROOT)
if not os.path.isdir(os.path.join(REPO_DIR, "src")):
    if os.path.isdir(REPO_DIR):
        sh(f"rm -rf {REPO_DIR}")
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")
else:
    # Already cloned: fetch the latest fixes/scripts without wiping anything.
    sh(f"git -C {REPO_DIR} fetch origin {BRANCH}")
    sh(f"git -C {REPO_DIR} reset --hard origin/{BRANCH}")
os.chdir(REPO_DIR)
sh("bash scripts/colab_setup.sh")

# Remember the movie source for the next cells.
if MOVIE_PATH:
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)
if MOVIE_URL:
    open("/content/movie_url.txt", "w").write(MOVIE_URL)
print("Setup complete. Repo:", os.getcwd())



### Cell 2 — Vision dependencies
Upgrades/verifies a Transformers build carrying the Qwen VL classes
(`Qwen2_5_VLForConditionalGeneration`, so `AutoProcessor` + `AutoModel` resolve
the vision model correctly), plus bitsandbytes (optional 4-bit) and Pillow.


In [ ]:

# @title 2) Vision deps (Qwen-VL transformers + pillow + gdown)
sh("bash scripts/colab_vision_setup.sh")
print("Vision deps ready.")


### Cell 3 — Get + validate the movie
Priority: `MOVIE_URL` (Drive share link, pasted in Cell 1) → `MOVIE_PATH` → upload.
The full movie is kept (`TRIM_TO_SEC = 0`); the next cell (3b) automatically picks
a 120s section that contains real dialogue/music.


In [ ]:

# @title 3) Get + validate movie file
import os, re, subprocess

TRIM_TO_SEC = 0  # @param {type:"number"} 0 = keep the FULL movie; Cell 3b auto-picks a dialogue window. Set >0 to force the first N seconds instead.

def _read(p):
    try:
        return open(p).read().strip()
    except FileNotFoundError:
        return ""

MOVIE_PATH = _read("/content/movie_path.txt")
MOVIE_URL = _read("/content/movie_url.txt")

# In a reused session, /content/movie_path.txt may still point at an old
# 120s trim from a previous TRIM_TO_SEC run. Recover the FULL original
# source so Cell 3b can pick a real dialogue window.
_orig = _read("/content/movie_original.txt")
if _orig and os.path.exists(_orig):
    MOVIE_PATH = _orig
elif os.path.exists("/content/movie_download") and os.path.getsize("/content/movie_download") > 1024 * 1024:
    MOVIE_PATH = "/content/movie_download"


# Tolerate a URL accidentally pasted into the MOVIE_PATH field
if MOVIE_PATH.startswith("http"):
    MOVIE_URL = MOVIE_URL or MOVIE_PATH
    MOVIE_PATH = ""

# Normalize any Drive URL to the uc?id= form gdown parses natively.
if MOVIE_URL:
    m = re.search(r"(?:file/d/|id=)([a-zA-Z0-9_-]{10,})", MOVIE_URL)
    if m:
        fid = m.group(1)
        MOVIE_URL = f"https://drive.google.com/uc?id={fid}&export=download"
        print("Using Drive file id:", fid)

if MOVIE_URL and not MOVIE_PATH:
    try:
        import gdown
    except ImportError:
        sh("pip install -q gdown")
        import gdown
    print("Downloading movie from Google Drive link:", MOVIE_URL)
    saved = gdown.download(MOVIE_URL, output="/content/movie_download", quiet=False)
    assert saved and os.path.exists(saved), "gdown failed to download the movie"
    MOVIE_PATH = saved
    size = os.path.getsize(MOVIE_PATH)
    print(f"Downloaded {size/1e6:.1f} MB to {MOVIE_PATH}")
    assert size > 1024 * 1024, (
        f"Downloaded file is only {size} bytes - this is the error page, not the "
        "movie. Make sure the Drive link points to a real video file shared as "
        "'Anyone with the link'."
    )
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)

if not MOVIE_PATH:
    from google.colab import files
    print("Uploading a movie from this machine (or set MOVIE_URL / Drive path above):")
    up = files.upload()
    MOVIE_PATH = list(up.keys())[0]
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)

MOVIE_PATH = os.path.abspath(os.path.expanduser(MOVIE_PATH))
print("Movie:", MOVIE_PATH, "exists:", os.path.exists(MOVIE_PATH))
assert os.path.exists(MOVIE_PATH), "Movie file not found"

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration,size",
     "-show_entries", "stream=codec_type,codec_name,width,height",
     "-of", "default=noprint_wrappers=1", MOVIE_PATH],
    capture_output=True, text=True)
print(probe.stdout or probe.stderr)
open("/content/movie_original.txt", "w").write(MOVIE_PATH)

assert "codec_type=video" in (probe.stdout or ""), (
    "The file at MOVIE_PATH is not a valid video. If using MOVIE_URL, make sure "
    "it is a real video file shared as 'Anyone with the link'."
)

if TRIM_TO_SEC and int(TRIM_TO_SEC) > 0:
    trimmed = "/content/movie_trimmed.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", MOVIE_PATH,
         "-t", str(int(TRIM_TO_SEC)), "-c", "copy", trimmed], check=True)
    MOVIE_PATH = trimmed
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)
    print(f"Trimmed to first {int(TRIM_TO_SEC)}s -> {MOVIE_PATH}")
else:
    open("/content/movie_path.txt", "w").write(MOVIE_PATH)
    print("Keeping full movie for Cell 3b:", MOVIE_PATH)

print("OK: valid video file.")


### Cell 3b — Auto-pick a dialogue window (no input needed)
Scans the FULL movie for silence (`silencedetect`), keeps the earliest loud run
of at least `WINDOW_SEC`, trims that exact window to `/content/movie_trimmed.mp4`,
and re-registers it so every later cell analyzes the same dialogue section.
Fully automatic on "Run all" — you only paste the movie link in Cell 1.


In [ ]:
# @title 3b) Auto-select a dialogue window (full movie required)
import os, re, subprocess

WINDOW_SEC = 120  # @param {type:"number"}

def _read(p):
    try:
        return open(p).read().strip()
    except FileNotFoundError:
        return ""

src = _read("/content/movie_original.txt") or open("/content/movie_path.txt").read().strip()
assert src and os.path.exists(src), "run Cell 3 first (paste the link in Cell 1)"

def _dur(path):
    return float(subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", path],
        capture_output=True, text=True).stdout.strip())

dur = _dur(src)
r = subprocess.run(
    ["ffmpeg", "-hide_banner", "-i", src,
     "-af", "silencedetect=n=-30dB:d=0.7", "-f", "null", "-"],
    capture_output=True, text=True)
marks = [float(x) for x in re.findall(r"silence_(?:start|end): ([0-9.]+)", r.stderr)]
marks.sort()

# loud intervals = complement of the [start,end] silence runs (treat 0 -> first silence as loud)
loud = []
if marks:
    start, state = 0.0, "loud"
    for m in marks:
        if state == "loud":            # m is a silence start -> close a loud interval
            if m > start:
                loud.append((start, m))
            state = "silent"
        else:                          # m is a silence end -> reopen loudness
            start = m
            state = "loud"
    if state == "loud" and dur > start:
        loud.append((start, dur))
else:
    loud = [(0.0, dur)] if dur > 0 else []
print(f"Duration: {dur/60:.1f} min | loud segments: {len(loud)}")
for a, b in loud[:10]:
    print(f"  loud {a/60:.1f}-{b/60:.1f} min ({b-a:.0f}s)")

W = int(WINDOW_SEC)
window = None
for a, b in loud:                      # earliest loud run long enough to fill the window
    if b - a >= W:
        window = (int(a), int(a) + W)
        break
if window is None:
    if loud:                           # center on the longest loud run
        a, b = max(loud, key=lambda p: p[1] - p[0])
        start = max(0, int(a + max(0, (b - a - W) / 2)))
    else:
        start = 0
    window = (start, start + W)
if dur <= W:
    window = (0, int(dur))
window = (window[0], min(window[1], int(dur)))

out = "/content/movie_trimmed.mp4"
subprocess.run(
    ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
     "-ss", str(window[0]), "-i", src, "-t", str(W), "-c", "copy", out],
    check=True)
open("/content/movie_path.txt", "w").write(out)
print(f"Chosen dialogue window: {window[0]//60}m {window[0]%60:02d}s -> +{W}s  ({out})")



### Cell 4 — Env vars + doctor
Enables the `colab-gpu` profile and **strict real-vision mode**. The director
and script are left off so this validation is focused on the vision layer only.


In [ ]:

# @title 4) Env vars + doctor (vision stage)
import os
os.environ["STUDIO_PROFILE"] = "colab-gpu"
os.environ["REQUIRE_REAL_VISION"] = "true"   # refuse heuristic fallback
os.environ["VISION_ENRICHER"] = "qwen3vl"
os.environ["VISION_MODEL"] = "Qwen/Qwen2.5-VL-3B-Instruct"  # 3B bf16 fits the free T4 (16GB) with no CPU offload / no OOM; switch to the 7B only on a 24GB+ GPU.
os.environ["VISION_DTYPE"] = "auto"  # bf16; the 3B model fits in the T4 without 4-bit quantization.
os.environ["VISION_MAX_FRAMES"] = "1" # keyframes per scene

import sys
sys.path.insert(0, "src")
from movie_understanding.enrich_factory import create_scene_enricher_from_env

enricher = create_scene_enricher_from_env()
print("Resolved scene enricher:", type(enricher).__name__, "name =", enricher.name)
print("Visually available:", enricher.available)
assert enricher.available, (
    "Strict vision mode requires CUDA + a Qwen-VL-capable transformers. "
    "Check the GPU (Runtime -> Change runtime type -> T4 GPU) and Cell 2."
)



### Cell 5 — Initialize the project
Registers the real movie (path only) under `data/<project-id>`.


In [ ]:

# @title 5) init project
import subprocess, re, sys
from pathlib import Path

MOVIE_PATH = Path(open("/content/movie_path.txt").read().strip()).expanduser().resolve()
assert MOVIE_PATH.exists(), "no movie was registered - run Cell 3 first"
out = subprocess.run(
    ["python", "src/main.py", "init", "--title", "Real Movie Vision Test",
     "--source", str(MOVIE_PATH)],
    capture_output=True, text=True)
print(out.stdout or out.stderr)
m = re.search(r"project ([0-9a-f-]{36})", out.stdout or "")
PROJECT_ID = m.group(1) if m else None
print("PROJECT_ID =", PROJECT_ID)
assert PROJECT_ID, "failed to init project"
open("/content/project_id.txt", "w").write(PROJECT_ID)



### Cell 6 — Transcript + scene detection (the inputs vision enriches)
Runs the same two understanding stages the orchestrator uses before enrichment:
WhisperX transcription (or the local stub) and PySceneDetect scene indexing.
For a 120s trim this is fast. We call the adapters directly, exactly like the
orchestrator does, so the vision stage consumes the same artifacts.


In [ ]:

# @title 6) transcription + scene detection
import sys
sys.path.insert(0, "src")
from pathlib import Path

PROJECT_ID = open("/content/project_id.txt").read().strip()
proj = Path("data") / PROJECT_ID
MOVIE_PATH = Path(open("/content/movie_path.txt").read().strip())

from transcription.adapter import transcribe
transcribe(proj, str(MOVIE_PATH))

from scene_indexing.adapter import build_scene_cards
try:
    build_scene_cards(proj, str(MOVIE_PATH))
except Exception as e:
    print(f"Scene detection failed: {e}")

import json
si = proj / "scenes" / "scene_index.json"
scenes = json.loads(si.read_text()) if si.exists() else []
print(f"Detected {len(scenes)} scene(s)")
for s in scenes[:12]:
    print(f"  {s['scene_id']} {s['start_sec']:.1f}-{s['end_sec']:.1f}s  transcript='{s.get('transcript','')[:60]}'")



### Cell 7 — Vision scene enrichment (THE validation)
Extracts a keyframe per scene, then runs the Qwen3-VL enricher through the
`MovieAnalyzer` (the exact code the editorial pipeline uses). This loads the
vision model **once** (shared class-level cache) and prompts it with each
scene's keyframe + transcript. Expect the first call to be slow (~1-2 min for
model load on a T4); subsequent scenes add a few seconds each.

After the analysis this cell also writes the derived artifacts:
`scene_index_v2.json`, `movie_memory/` (bundle), and
`reports/movie_understanding_report.md`.


In [ ]:

# @title 7) build movie index with vision enrichment (real Qwen3-VL)
import sys, time, json
sys.path.insert(0, "src")
from pathlib import Path

PROJECT_ID = open("/content/project_id.txt").read().strip()
proj = Path("data") / PROJECT_ID

from movie_understanding.analyzer import MovieAnalyzer
from movie_understanding.scene_analyzer import HeuristicSceneEnricher
from movie_understanding.vision_enricher import Qwen3VLEnricher

t0 = time.time()
enricher = Qwen3VLEnricher(
    model=os.environ.get("VISION_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct"),
    dtype=os.environ.get("VISION_DTYPE", "auto"),
    max_frames=int(os.environ.get("VISION_MAX_FRAMES", "1")),
    strict=True,
)
idx = MovieAnalyzer(scene_enricher=enricher, attach_keyframes=True).analyze(proj)
print(f"--- vision enrichment wall time: {time.time()-t0:.1f}s ---\n")
print("Vision model load (s):", enricher.model_load_time_sec)
print("Vision model generate (s):", enricher.last_generation_time_sec)
print("Provenance:", idx.get("provenance"))

# Derived artifacts the director consumes (written by the analyzer; the report
# is a human-readable audit of the same data).
from movie_understanding.artifacts import (
    write_movie_understanding_report,
)
rep = write_movie_understanding_report(proj)
print("Report:", rep)
print("scene_index_v2.json:", (proj / "scene_index_v2.json").exists())
print("movie_memory/:", (proj / "movie_memory").is_dir())
print("semantic_index.json:", (proj / "semantic_index.json").exists())



### Cell 7b — Natural-language retrieval evaluation
Runs the evaluation queries (see `scripts/evaluate_retrieval.py`) against the
built semantic index and writes `reports/retrieval_evaluation.json` + `.md`.
The automated fields are the scene ids / timestamps / scores the index
actually returned; `human_assessment` (GOOD / PARTIAL / WRONG) is left blank
for you to fill by inspecting the top results manually (Cell 9 shows the
keyframes next to their descriptions).

In [ ]:

# @title 7b) Retrieval evaluation (semantic index -> reports)
import subprocess, json
from pathlib import Path

PROJECT_ID = open("/content/project_id.txt").read().strip()
proj = Path("data") / PROJECT_ID

# Evaluation queries live in scripts/evaluate_retrieval.py (not production code).
res = subprocess.run(
    ["python", "scripts/evaluate_retrieval.py", "--project", str(proj)],
    capture_output=True, text=True)
print(res.stdout or res.stderr)
assert res.returncode == 0, "retrieval evaluation failed"
recs = json.loads((proj / "reports" / "retrieval_evaluation.json").read_text())
print(f"Wrote {len(recs['queries'])} query records -> reports/retrieval_evaluation.json")



### Cell 7c — Temporal probe (optional)
For a few scenes, re-enriches with multiple keyframes spread across the scene
window and asks the model to order the visual events with *approximate*
timestamps (character enters, sits, object appears, close-up, dialogue,
reaction, leaves). Honest failure mode: if the model cannot localize an event
to a time, it says so instead of guessing — document that rather than
pretending temporal localization is precise.

In [ ]:

# @title 7c) Temporal probe: ordered visual events with approx timestamps
import sys, json, time
sys.path.insert(0, "src")
from pathlib import Path

PROJECT_ID = open("/content/project_id.txt").read().strip()
proj = Path("data") / PROJECT_ID

TEMPORAL_PROBE_SCENES = []  # @param leave empty to probe up to 3 longest scenes
TEMPORAL_FRAMES = 4         # @param keyframes spread across each scene window

idx = json.loads((proj / "movie_index.json").read_text())
scenes = idx["scenes"]
if TEMPORAL_PROBE_SCENES:
    targets = [s for s in scenes if s["scene_id"] in TEMPORAL_PROBE_SCENES]
else:
    targets = sorted(scenes, key=lambda s: -(s.get("end_sec", 0) - s.get("start_sec", 0)))[:3]

from movie_understanding.vision_enricher import Qwen3VLEnricher
from movie_understanding.keyframes import extract_scene_keyframes

enricher = Qwen3VLEnricher(
    model=os.environ.get("VISION_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct"),
    dtype=os.environ.get("VISION_DTYPE", "auto"),
    max_frames=TEMPORAL_FRAMES,
    strict=True,
)
import movie_understanding.vision_enricher as _ve

meta = json.loads((proj / "project_meta.json").read_text()) if (proj / "project_meta.json").exists() else {}
kf_dir = proj / "scenes" / "keyframes_temporal"
results = []
for scene in targets:
    sid = scene["scene_id"]
    try:
        frames = extract_scene_keyframes(
            meta.get("source_path") or idx.get("source_path"),
            scene["start_sec"], scene["end_sec"], kf_dir, scene_id=sid,
            max_frames=TEMPORAL_FRAMES)
        scene = dict(scene); scene["key_frames"] = [str(f) for f in frames]
        out = enricher.probe_temporal(scene, [])
    except Exception as e:
        out = {"scene_id": sid, "ok": False, "reason": str(e)}
    results.append(out)
    print(json.dumps(out, indent=2, ensure_ascii=False)[:1200])

(proj / "reports").mkdir(parents=True, exist_ok=True)
(proj / "reports" / "temporal_probe.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote reports/temporal_probe.json")



### Cell 8 — Inspect the vision-filled scene cards
Shows `location` / `actions` / `visual_description` / `themes` / `mood` and
per-field provenance for each scene. **Success is not "process finished"**:
the values must be real visual observations grounded in each frame, and
`provenance` for those fields must read `qwen3vl` (not `unavailable`).


In [ ]:

# @title 8) Inspect vision-enriched scene cards
import sys, json
sys.path.insert(0, "src")
from pathlib import Path

PROJECT_ID = open("/content/project_id.txt").read().strip()
proj = Path("data") / PROJECT_ID
idx = json.loads((proj / "movie_index.json").read_text())

vision_filled = 0
for scene in idx["scenes"]:
    story = scene["story"]
    print(f"= {scene['scene_id']} ({scene['start_sec']:.1f}-{scene['end_sec']:.1f}s)")
    print(f"  location: {story.get('location')}")
    print(f"  actions:  {story.get('actions')}")
    print(f"  objects:  {story.get('objects')}")
    print(f"  visual:   {story.get('visual_description')}")
    print(f"  events:   {story.get('visual_events')}")
    print(f"  cues:     {story.get('emotional_cues')}")
    print(f"  themes:   {story.get('themes')}")
    print(f"  mood:     {story.get('mood')}")
    print(f"  cinematography: {story.get('cinematography')}")
    print(f"  confidence: {story.get('confidence')}")
    prov = story.get("provenance", {})
    cv = (prov.get("location"), prov.get("visual_description"), prov.get("mood"))
    if all(v == "qwen3vl" for v in cv):
        vision_filled += 1
    print()

total = len(idx["scenes"])
print(f"Vision-filled scenes: {vision_filled}/{total}")
assert vision_filled == total, "Some scenes did not get vision enrichment (strict mode would have raised; check Cell 7)"
print("OK: every scene has real Qwen3-VL visual fields.")



### Cell 9 — Visual QA: show a keyframe next to its description
Renders one keyframe per scene alongside the model's own `visual_description`
so you can eyeball whether the model is describing the actual frame.


In [ ]:

# @title 9) Keyframe + description QA
import sys, json
sys.path.insert(0, "src")
from pathlib import Path
from IPython.display import display, Image, HTML

PROJECT_ID = open("/content/project_id.txt").read().strip()
proj = Path("data") / PROJECT_ID
idx = json.loads((proj / "movie_index.json").read_text())

kf_dir = proj / "scenes" / "keyframes"
for scene in idx["scenes"]:
    sid = scene["scene_id"]
    frames = list(kf_dir.glob(f"{sid}_*.jpg"))
    if not frames:
        continue
    story = scene["story"]
    desc = (story.get("visual_description") or "(no description)")[:300]
    print(f"{sid}: {desc}")
    display(Image(str(frames[0]), width=420))
    print()



### How to evaluate the vision stage (before the next milestone)
A green run is **not** success. For a sample of 10+ scenes, score:

1. **Location** — Is the setting correct (indoor/outdoor, type of place)?
2. **Actions** — Does it capture the actual on-screen action?
3. **Objects** — Are the objects actually in the frame, not hallucinated?
4. **Visual description** — Grounded in this frame, specific, not generic?
5. **Visual events** — Do the approx-timestamped events match what happens?
6. **Emotional cues** — Are they visible in the frame vs. inferred from script?
7. **Themes** — Useful for the editorial director's argument, not surface noise?
8. **Mood** — Reasonable given the image + ambient audio?
9. **Cinematography** — Shot size/camera/lighting described correctly?
10. **Confidence** — Does the model's self-assessment track correctness?
11. **Observation vs interpretation** — Does the card distinguish what is
    visibly there from what the model infers?
12. **Failure mode** — Where does it drift? (crowds, fast motion, title cards,
    abstract shots, end credits)

Then run Cells 7b/7c and score retrieval (`GOOD`/`PARTIAL`/`WRONG`) and
temporal localization honestly in `reports/`. Log what you observe. If vision
enrichment is solid, the next natural step is a full editorial GPU run that
consumes these visuals (`VISION_ENRICHER=qwen3vl EDITORIAL_MODE=true`).


### Optional Cell 10 — Full editorial run WITH vision fields
After vision enrichment passes QA, run the whole editorial pipeline so the
evidence-driven script can reference visuals. Same as
`colab_editorial_gpu.ipynb`, but with `VISION_ENRICHER=qwen3vl` so
`movie_index.json` carries the vision story cards the retrieval consumes.


In [ ]:

# @title 10) [optional] full editorial pipeline with vision-rich movie index
# Uncomment and adjust (needs a TTS provider + director as in colab_editorial_gpu):
# import os
# os.environ["REQUIRE_REAL_LLM"] = "true"
# os.environ["REQUIRE_REAL_TTS"] = "true"
# os.environ["DIRECTOR_PROVIDER"] = "qwen"
# os.environ["DIRECTOR_MODEL"] = "Qwen/Qwen3-4B-Instruct-2507"
# os.environ["TTS_DEVICE"] = "cuda"
# os.environ["TTS_PROVIDER"] = "kokoro"
# os.environ["EDITORIAL_MODE"] = "true"
# os.environ["EDITORIAL_TARGET_SEC"] = "90"
# os.environ["VISION_ENRICHER"] = "qwen3vl"
# os.environ["REQUIRE_REAL_VISION"] = "true"
# import subprocess
# PROJECT_ID = open("/content/project_id.txt").read().strip()
# out = subprocess.run(["python", "src/main.py", "run", "--project-id", PROJECT_ID], timeout=7200)
# assert out.returncode == 0, "editorial run failed"
# print("Full editorial run ready: renders/final_render.mp4")
